In [1]:
import time
from oggm import cfg, utils, workflow, tasks, graphics
from oggm.sandbox import ioggm_dynamic_spinup
from oggm.core.sia2d import IGM_Model2D, compute_2d_quantiles # import the IGM_Model2D
from datetime import datetime
import os
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
from oggm.shop import gcm_climate
from oggm.core.massbalance import (DistributedMassBalance,
                                   MonthlyTIModel)
import pandas as pd


/Users/afisc/igm_venv/bin/python


## volume or area minimisation:

In [2]:
# min_for = 'area'
min_for = 'volume'

In [3]:
cfg.initialize(logging_level='WARNING')

ts = datetime.now().strftime("%Y%m%d-%H%M%S")
cfg.PATHS['working_dir'] = os.path.join('/Users/afisc/instructed-oggm/experiments_working_dir/06_11/OGGM_meltf_and_gcm/{}__{}'.format(ts, min_for))


2026-06-11 12:38:30: oggm.cfg: Reading default parameters from the OGGM `params.cfg` configuration file.
2026-06-11 12:38:30: oggm.cfg: Multiprocessing switched OFF according to the parameter file.
2026-06-11 12:38:30: oggm.cfg: Multiprocessing: using all available processors (N=8)


## choose the glacier to work with

In [4]:
# Hintereisferner and a 'bad' glacier in terms of the dynamic spinup
# rgi_ids = ['RGI60-11.00897', 'RGI60-11.00275']

## Pick a glacier
rgi_ids = ['RGI60-11.01450']  # This is Aletsch
# rgi_ids = ['RGI60-11.00897']  # This is Hintereisferner
# rgi_ids = ['RGI60-11.03638']  # This is Argentiere

## defining and loading the pre-processed gdir data

In [5]:
# We use a recent gdir setting, calibated on a glacier per glacier basis
base_url = ('https://cluster.klima.uni-bremen.de/~oggm/gdirs/oggm_v1.6/'
            'L3-L5_files/2023.3/elev_bands/W5E5/')
# base_url = ('https://cluster.klima.uni-bremen.de/~oggm/gdirs/oggm_v1.6/L3-L5_files/2023.3/elev_bands/W5E5_spinup')


In [6]:
gdirs = workflow.init_glacier_directories(rgi_ids,from_prepro_level=3, prepro_border=80, prepro_base_url=base_url)



2026-06-11 12:38:31: oggm.workflow: init_glacier_directories from prepro level 3 on 1 glaciers.
2026-06-11 12:38:31: oggm.workflow: Execute entity tasks [gdir_from_prepro] on 1 glaciers


In [7]:
gdir = gdirs[0]

In [8]:
# add consensus now as well, as we use a different prepro
from oggm.shop import bedtopo

workflow.execute_entity_task(bedtopo.add_consensus_thickness, gdir);

2026-06-11 12:38:31: oggm.workflow: Execute entity tasks [add_consensus_thickness] on 1 glaciers


In [9]:
thick = tasks.distribute_thickness_per_altitude(gdir)

In [10]:
cfg.PARAMS['store_fl_diagnostics'] = True
spinup_start_yr = 1979
output_filesuffix = f'_meltf_{min_for}'
kwargs_run_function = {
    'minimise_for': min_for,
    'store_diagnostics_spinup': True
}

tasks.run_dynamic_melt_f_calibration(gdir,
                                     ys=spinup_start_yr,  # When to start the spinup
                                     ye=2020,  # When the simulation should stop
                                     output_filesuffix=output_filesuffix, # Where to write the output
                                     kwargs_run_function=kwargs_run_function,
                                    );


2026-06-11 12:38:32: oggm.cfg: PARAMS['store_fl_diagnostics'] changed from `False` to `True`.


## Load gcm data

In [11]:
temperature_scenarios = False

In [12]:
# load CMIP5 + CMIP6 metadata
gcms_cmip6 = pd.read_csv('/Users/afisc/atmo_master/rgi_job/oggm/cmip_data/cmip6/all_gcm_list.csv', index_col=0)
gcms_cmip5 = pd.read_csv('/Users/afisc/atmo_master/rgi_job/oggm/cmip_data/cmip5-ng/all_gcm_list.csv', index_col=0)

In [13]:
def remote_path(path):
    base_local = "/home/www/oggm/"
    base_remote = "https://cluster.klima.uni-bremen.de/~oggm/"

    if path.startswith(base_local):
        return path.replace(base_local, base_remote, 1)
    else:
        raise ValueError(f"Path does not start with {base_local}")

In [14]:
if not temperature_scenarios:
    # set the ssp scenarios here
    scenarios = ['ssp126', 
                 'ssp245', 
                 'ssp585']


    def get_models_per_scenario(scenario):
        return np.unique(gcms_cmip6[gcms_cmip6.ssp == scenario].gcm.values)


    gcms_per_scenario = {}
    for scenario in scenarios:
        gcms_per_scenario[scenario] = get_models_per_scenario(scenario)
        # if is_notebook:
        #     gcms_per_scenario[scenario] = [gcms_per_scenario[scenario][0]]

    # iterate through the scenarios        
    for scenario in gcms_per_scenario:
        for gcm in gcms_per_scenario[scenario]:
            rid = f'{gcm}_{scenario}'

            select_gcm = np.array([g.upper() for g in gcms_cmip6.gcm]) == gcm.upper()
            select_ssp = gcms_cmip6.ssp == scenario
            selected_run = gcms_cmip6[select_gcm & select_ssp]
            ft = selected_run[selected_run['var'] == 'tas'].path.values[0]
            fp = selected_run[selected_run['var'] == 'pr'].path.values[0]

            ft = utils.file_downloader(remote_path(ft))
            fp = utils.file_downloader(remote_path(fp))
            # print(ft)
            # print(fp)
            # bias correct them
            workflow.execute_entity_task(gcm_climate.process_cmip_data, gdirs,
                                         year_range=('2000', '2019'),
                                         filesuffix=rid,  # recognize the climate file for later
                                         fpath_temp=ft,  # temperature projections
                                         fpath_precip=fp,  # precip projections
                                         );


2026-06-11 12:38:39: oggm.workflow: Execute entity tasks [process_cmip_data] on 1 glaciers
2026-06-11 12:38:45: oggm.workflow: Execute entity tasks [process_cmip_data] on 1 glaciers
2026-06-11 12:38:50: oggm.workflow: Execute entity tasks [process_cmip_data] on 1 glaciers
2026-06-11 12:38:53: oggm.workflow: Execute entity tasks [process_cmip_data] on 1 glaciers
2026-06-11 12:38:55: oggm.workflow: Execute entity tasks [process_cmip_data] on 1 glaciers
2026-06-11 12:38:57: oggm.workflow: Execute entity tasks [process_cmip_data] on 1 glaciers
2026-06-11 12:39:04: oggm.workflow: Execute entity tasks [process_cmip_data] on 1 glaciers
2026-06-11 12:39:07: oggm.workflow: Execute entity tasks [process_cmip_data] on 1 glaciers
2026-06-11 12:39:08: oggm.workflow: Execute entity tasks [process_cmip_data] on 1 glaciers
2026-06-11 12:39:10: oggm.workflow: Execute entity tasks [process_cmip_data] on 1 glaciers
2026-06-11 12:39:13: oggm.workflow: Execute entity tasks [process_cmip_data] on 1 glaciers

In [15]:
for ssp_scenario, gcms in gcms_per_scenario.items():
    gcms = gcms.astype(str)
    gcms_per_scenario[ssp_scenario] = np.char.add(gcms, f"_{ssp_scenario}").tolist()
    scenario_suffixes = gcms_per_scenario


## GCM run 2020-2100


In [16]:
for scenario in scenario_suffixes.values():
    for scenario_suffix in scenario:
        workflow.execute_entity_task(tasks.run_from_climate_data, gdirs,
                                     ye=2100,
                                     climate_filename='gcm_data',  # use gcm_data, not climate_historical
                                     climate_input_filesuffix=scenario_suffix,  # use the chosen scenario
                                     init_model_filesuffix=output_filesuffix,  # this is important! Start from 2020 glacier
                                     output_filesuffix=scenario_suffix,  # recognize the run for later
                                    );
        

2026-06-11 12:40:19: oggm.workflow: Execute entity tasks [run_from_climate_data] on 1 glaciers
2026-06-11 12:40:20: oggm.workflow: Execute entity tasks [run_from_climate_data] on 1 glaciers
2026-06-11 12:40:20: oggm.workflow: Execute entity tasks [run_from_climate_data] on 1 glaciers
2026-06-11 12:40:20: oggm.workflow: Execute entity tasks [run_from_climate_data] on 1 glaciers
2026-06-11 12:40:20: oggm.workflow: Execute entity tasks [run_from_climate_data] on 1 glaciers
2026-06-11 12:40:21: oggm.workflow: Execute entity tasks [run_from_climate_data] on 1 glaciers
2026-06-11 12:40:21: oggm.workflow: Execute entity tasks [run_from_climate_data] on 1 glaciers
2026-06-11 12:40:21: oggm.workflow: Execute entity tasks [run_from_climate_data] on 1 glaciers
2026-06-11 12:40:21: oggm.workflow: Execute entity tasks [run_from_climate_data] on 1 glaciers
2026-06-11 12:40:22: oggm.workflow: Execute entity tasks [run_from_climate_data] on 1 glaciers
2026-06-11 12:40:22: oggm.workflow: Execute entity

In [17]:
for scenario, suffix_list in scenario_suffixes.items():
    workflow.execute_entity_task(tasks.compute_diagnostics_quantiles,
                                 gdirs,
                                 input_filesuffixes=suffix_list,
                                 diag_file='model_diagnostics',
                                 quantiles=[0.5, 0.25, 0.75], 
                                 output_filesuffix=f'_{scenario}_quantiles'
                                 )
    workflow.execute_entity_task(tasks.compute_diagnostics_quantiles,
                                 gdirs,
                                 input_filesuffixes=suffix_list,
                                 diag_file='fl_diagnostics',
                                 quantiles=[0.5, 0.25, 0.75], 
                                 output_filesuffix=f'_{scenario}_quantiles'
                                 )

2026-06-11 12:40:32: oggm.workflow: Execute entity tasks [compute_diagnostics_quantiles] on 1 glaciers
2026-06-11 12:40:32: oggm.workflow: Execute entity tasks [compute_diagnostics_quantiles] on 1 glaciers
2026-06-11 12:40:36: oggm.workflow: Execute entity tasks [compute_diagnostics_quantiles] on 1 glaciers
2026-06-11 12:40:36: oggm.workflow: Execute entity tasks [compute_diagnostics_quantiles] on 1 glaciers
2026-06-11 12:40:40: oggm.workflow: Execute entity tasks [compute_diagnostics_quantiles] on 1 glaciers
2026-06-11 12:40:41: oggm.workflow: Execute entity tasks [compute_diagnostics_quantiles] on 1 glaciers
